# PCBA 缺陷数据集 EDA 分析
> 数据集: PKU PCB 缺陷数据集 (10,668 张 / 6 类缺陷)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / 'data' / 'raw' / 'PCB_DATASET'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 数据集概览

In [ ]:
# 统计目录结构
def explore_dir(path, max_depth=2):
    for root, dirs, files in os.walk(path):
        depth = root[len(str(path)):].count(os.sep)
        if depth <= max_depth:
            print(f"{'  ' * depth}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)")

import os
explore_dir(DATA_RAW)

## 2. 标注文件统计

In [ ]:
import xml.etree.ElementTree as ET

xml_files = sorted(DATA_RAW.rglob('*.xml'))
print(f'标注文件总数: {len(xml_files)}')

# 统计每个 XML 的类别分布和图像尺寸
class_counts = Counter()
sizes = []
boxes_per_image = []

for xml_path in xml_files:
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # 图像尺寸
    size = root.find('size')
    if size is not None:
        w = int(size.find('width').text)
        h = int(size.find('height').text)
        sizes.append((w, h))

    # 标注框统计
    n_boxes = len(root.findall('object'))
    boxes_per_image.append(n_boxes)

    for obj in root.findall('object'):
        cls_name = obj.find('name').text
        class_counts[cls_name] += 1

print(f'\n总标注框数: {sum(class_counts.values())}')
print(f'平均每图标注框: {np.mean(boxes_per_image):.2f}')
print(f'\n类别分布:')
for name, count in class_counts.most_common():
    print(f'  {name}: {count} ({count/sum(class_counts.values())*100:.1f}%)')

if class_counts:
    max_cls = class_counts.most_common(1)[0][1]
    min_cls = class_counts.most_common()[-1][1]
    print(f'\n类别不均衡度: {max_cls/min_cls:.1f}:1')

## 3. 可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 类别分布
names = [n for n, _ in class_counts.most_common()]
counts = [c for _, c in class_counts.most_common()]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
axes[0].barh(names, counts, color=colors)
axes[0].set_title('缺陷类别分布')
axes[0].set_xlabel('标注框数量')

# 每图标注框数分布
axes[1].hist(boxes_per_image, bins=20, color='#45B7D1', edgecolor='white')
axes[1].set_title('每图标注框数量分布')
axes[1].set_xlabel('标注框数')
axes[1].set_ylabel('图片数')

# 图像尺寸分布
if sizes:
    ws, hs = zip(*sizes)
    axes[2].scatter(list(ws)[:500], list(hs)[:500], alpha=0.3, s=5)
    axes[2].set_title('图像尺寸分布 (前500张)')
    axes[2].set_xlabel('宽度 (px)')
    axes[2].set_ylabel('高度 (px)')

plt.tight_layout()
plt.show()

if sizes:
    ws_arr = np.array(ws)
    hs_arr = np.array(hs)
    print(f'\n图像尺寸统计:')
    print(f'  宽度: 均值={ws_arr.mean():.0f}, 中位数={np.median(ws_arr):.0f}, min={ws_arr.min()}, max={ws_arr.max()}')
    print(f'  高度: 均值={hs_arr.mean():.0f}, 中位数={np.median(hs_arr):.0f}, min={hs_arr.min()}, max={hs_arr.max()}')

## 4. 结论
- 类别分布: _____ (将上方数字填入项目规划.md)
- 每图平均框数: _____
- 类别不均衡度: _____
- 图像尺寸均值: _____ × _____